<a href="https://colab.research.google.com/github/praveenkyadav3103-star/data-analysis/blob/master/GroupDNA_PRAVEEN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 GroupDNA — WhatsApp Group Behaviour Analyzer

### CSE 2026 — Code, Canteen & Chaos

| **Project Detail** |    **Information** |
|---|-----|
| **Developed by** | Praveen Kumar Yadav |
| **Batch** | Data Analytics |
| **Project Type** | Data Analytics Minor Project |
| **Technologies Used** | Python, NumPy, File I/O and Datetime |
| **Dataset** | Synthetic Indian College WhatsApp Group Chat |
| **Submitted To** | Adyapan Academy |
| **Submission Date** | 27August 2026 |
---

## Project Objective

GroupDNA analyzes a raw WhatsApp group-chat export to identify group activity, messaging patterns, frequently used words, response behaviour, silent streaks and participant personality archetypes.

The project is developed using Python fundamentals and NumPy without using Pandas, Matplotlib, regular expressions or pre-built WhatsApp-analysis libraries.

In [1]:
# Import Required Libraries

import numpy as np
from datetime import datetime, timedelta
print("GroupDNA environment is ready!")

GroupDNA environment is ready!


In [ ]:
# Upload WhatsApp Chat Dataset in Google Colab
from google.colab import files
uploaded = files.upload()
print("Dataset uploaded successfully!")

In [ ]:
# Read WhatsApp Chat Dataset
file_name = "cse_2026_survival_squad_10_members.txt"
with open( file_name,"r", encoding="utf-8"
) as file:
    lines = file.readlines()
print("Dataset read successfully!")
print("Total Raw Lines:", len(lines))

Dataset read successfully!
Total Raw Lines: 3125


In [ ]:
# Display First 10 Raw Chat Lines
print("FIRST 10 RAW CHAT LINES")
print("=" * 70)
for i in range(10):
    print(lines[i])

FIRST 10 RAW CHAT LINES
01/02/24, 11:00 - Priya created group "CSE 2026 - Code, Canteen & Chaos"

01/02/24, 11:01 - Priya added Anchal

01/02/24, 11:02 - Priya added Rahul

01/02/24, 11:03 - Priya added Aman

01/02/24, 11:04 - Priya added Karan

01/02/24, 11:05 - Priya added Neha

01/02/24, 11:06 - Priya added Vikas

01/02/24, 11:07 - Priya added Riya

01/02/24, 11:08 - Priya added Naman

01/02/24, 11:09 - Priya added Meera



Feature 1 – Chat Parser

In [ ]:
#  Chat Parser
messages = []
system_messages = 0
media_omitted = 0
deleted_messages = 0
continuation_lines = 0
for line in lines:
    line = line.strip()
    # Skip empty lines
    if line == "":
        continue
    # Check whether line starts with DD/MM/YY
    starts_with_date = (
        len(line) >= 8
        and line[0:2].isdigit()
        and line[2] == "/"
        and line[3:5].isdigit()
        and line[5] == "/"
        and line[6:8].isdigit())
    # Handle multi-line message
    if not starts_with_date:
        continuation_lines += 1
        if len(messages) > 0:
            messages[-1]["text"] = (
                messages[-1]["text"]
                + " "
                + line
            )
        continue
    # Separate timestamp and remaining content
    if " - " not in line:
        continue
    parts = line.split(" - ", 1)
    timestamp = parts[0]
    remaining = parts[1]
    # System message does not contain sender separator
    if ": " not in remaining:
        system_messages += 1
        continue
    # Separate sender and message text
    parts2 = remaining.split(": ", 1)
    sender = parts2[0].strip()
    text = parts2[1].strip()
    # Count and skip media messages
    if text == "<Media omitted>":
        media_omitted += 1
        continue
    # Count and skip deleted messages
    if text.lower() == "this message was deleted":
        deleted_messages += 1
        continue
    # Store valid message
    messages.append({
        "timestamp": timestamp,
        "sender": sender,
        "text": text
    })
print("=" * 60)
print("PARSER SUMMARY")
print("=" * 60)
print("Total Raw Lines   :", len(lines))
print("Real Messages     :", len(messages))
print("System Messages   :", system_messages)
print("Media Omitted     :", media_omitted)
print("Deleted Messages  :", deleted_messages)
print("Continuation Lines:", continuation_lines)
print("=" * 60)

PARSER SUMMARY
Total Raw Lines   : 3125
Real Messages     : 3067
System Messages   : 11
Media Omitted     : 30
Deleted Messages  : 15
Continuation Lines: 2


In [ ]:
# Display First 5 Parsed Messages
print("FIRST 5 PARSED MESSAGES")
print("=" * 60)
for i in range(5):
    print(messages[i])

FIRST 5 PARSED MESSAGES
{'timestamp': '01/02/24, 11:30', 'sender': 'Anchal', 'text': 'sir attendance le rahe hain, proxy chahiye to roll number jaldi bhejo'}
{'timestamp': '01/02/24, 11:33', 'sender': 'Meera', 'text': 'mera 42 hai please present bol dena'}
{'timestamp': '01/02/24, 11:34', 'sender': 'Naman', 'text': 'meri attendance update hui kya?'}
{'timestamp': '01/02/24, 11:37', 'sender': 'Anchal', 'text': 'Naman tum present ho, Meera ki proxy try karti hu'}
{'timestamp': '01/02/24, 11:40', 'sender': 'Neha', 'text': 'MERA NAAM MISS MAT KARNA!!'}


In [ ]:
# Display Last 5 Parsed Messages
print("LAST 5 PARSED MESSAGES")
print("=" * 70)
for i in range(len(messages) - 5, len(messages)):
    print(messages[i])

LAST 5 PARSED MESSAGES
{'timestamp': '31/03/24, 23:08', 'sender': 'Aman', 'text': 'koi online hai kya'}
{'timestamp': '31/03/24, 23:11', 'sender': 'Rahul', 'text': 'haan bhai'}
{'timestamp': '31/03/24, 23:12', 'sender': 'Aman', 'text': 'code me error aa raha hai aur neend bhi nahi aa rahi'}
{'timestamp': '31/03/24, 23:15', 'sender': 'Riya', 'text': 'lol error ko bhi midnight company chahiye'}
{'timestamp': '31/03/24, 23:16', 'sender': 'Aman', 'text': 'raat ko debugging ka alag hi scene hai'}


In [ ]:
# Validate Parsed Timestamps
valid_timestamps = 0
for msg in messages:
    datetime.strptime(
        msg["timestamp"],
        "%d/%m/%y, %H:%M"
    )
    valid_timestamps += 1
print("TIMESTAMP VALIDATION")
print("=" * 50)
print("Valid Timestamps :", valid_timestamps)
print("Parsed Messages  :", len(messages))
if valid_timestamps == len(messages):
    print("Status           : All timestamps are valid")
else:
    print("Status           : Some timestamps are invalid")

TIMESTAMP VALIDATION
Valid Timestamps : 3067
Parsed Messages  : 3067
Status           : All timestamps are valid


Feature 2 – Group Overview

In [ ]:
# Find Participants
participants = []
for msg in messages:
    sender = msg["sender"]
    if sender not in participants:
        participants.append(sender)
print("PARTICIPANTS")
print("=" * 60)
for person in participants:
    print(person)
print("=" * 60)
print("Total Participants:", len(participants))

PARTICIPANTS
Anchal
Meera
Naman
Neha
Riya
Rahul
Priya
Aman
Karan
Vikas
Total Participants: 10


In [ ]:
# Count Messages Per Person
message_count = {}
for msg in messages:
    sender = msg["sender"]
    if sender in message_count:
        message_count[sender] += 1
    else:
        message_count[sender] = 1
print("MESSAGES PER PERSON")
print("=" * 50)
for person in participants:
    print(f"{person:<10} :",message_count[person])
print("=" * 50)

MESSAGES PER PERSON
Anchal     : 306
Meera      : 344
Naman      : 393
Neha       : 158
Riya       : 318
Rahul      : 644
Priya      : 363
Aman       : 357
Karan      : 174
Vikas      : 10


In [ ]:
# Find Chat Date Range
first_date = datetime.strptime(
    messages[0]["timestamp"],"%d/%m/%y, %H:%M")
last_date = datetime.strptime(
    messages[-1]["timestamp"],"%d/%m/%y, %H:%M")
chat_start_date = first_date.date()
chat_end_date = last_date.date()
total_days = (chat_end_date - chat_start_date).days + 1
print("CHAT DATE RANGE")
print("=" * 50)
print("Start Date :",first_date.strftime("%d %B %Y"))
print("End Date   :", last_date.strftime("%d %B %Y"))
print( "Total Days :",total_days)
print("=" * 50)

CHAT DATE RANGE
Start Date : 01 February 2024
End Date   : 31 March 2024
Total Days : 60


In [ ]:
# Function to Return Message Count
def get_message_count(item):
    return item[1]
# Display Group Participation
print("GROUP PARTICIPATION")
print("=" * 60)
sorted_message_count = sorted(message_count.items(),
    key=get_message_count,
    reverse=True
)
for person, count in sorted_message_count:
    percentage = (count / len(messages)) * 100
    print( f"{person:<10} : "
        f"{count:<4} messages "
        f"({percentage:.2f}%)"
    )
print("=" * 60)

GROUP PARTICIPATION
Rahul      : 644  messages (21.00%)
Naman      : 393  messages (12.81%)
Priya      : 363  messages (11.84%)
Aman       : 357  messages (11.64%)
Meera      : 344  messages (11.22%)
Riya       : 318  messages (10.37%)
Anchal     : 306  messages (9.98%)
Karan      : 174  messages (5.67%)
Neha       : 158  messages (5.15%)
Vikas      : 10   messages (0.33%)


In [ ]:
# Find Most Active and Least Active Person
most_active = max(message_count,
    key=message_count.get
)
least_active = min(
    message_count,
    key=message_count.get
)
print("ACTIVITY LEADERS")
print("=" * 50)
print("Most Active Person  :",most_active,
    "-",
    message_count[most_active],
    "messages"
)
print( "Least Active Person :",least_active,"-",
 message_count[least_active],
    "messages"
)
print("=" * 50)

ACTIVITY LEADERS
Most Active Person  : Rahul - 644 messages
Least Active Person : Vikas - 10 messages


In [ ]:
# Per-Person Statistics

person_total_words = {}
story_average = {}

print("=" * 60)
print("PER-PERSON STATISTICS")
print("=" * 60)
for person in participants:
    total_messages = message_count[person]
    total_words = 0
    for msg in messages:
        if msg["sender"] == person:
            words = msg["text"].split()
            total_words += len(words)
    person_total_words[person] = total_words
    if total_messages > 0:
        average_length = (
            total_words / total_messages
        )
    else:
        average_length = 0
    story_average[person] = average_length
    print()
    print("Person             :", person)
    print("Messages           :", total_messages)
    print("Total Words        :", total_words)
    print(
        "Avg Message Length :",
        round(average_length, 2),
        "words"
    )
print("=" * 60)

PER-PERSON STATISTICS

Person             : Anchal
Messages           : 306
Total Words        : 2748
Avg Message Length : 8.98 words

Person             : Meera
Messages           : 344
Total Words        : 2990
Avg Message Length : 8.69 words

Person             : Naman
Messages           : 393
Total Words        : 1985
Avg Message Length : 5.05 words

Person             : Neha
Messages           : 158
Total Words        : 1018
Avg Message Length : 6.44 words

Person             : Riya
Messages           : 318
Total Words        : 2338
Avg Message Length : 7.35 words

Person             : Rahul
Messages           : 644
Total Words        : 1552
Avg Message Length : 2.41 words

Person             : Priya
Messages           : 363
Total Words        : 3392
Avg Message Length : 9.34 words

Person             : Aman
Messages           : 357
Total Words        : 2901
Avg Message Length : 8.13 words

Person             : Karan
Messages           : 174
Total Words        : 5673
Avg Message L

Feature 3 – Most Active Day and Hour

In [ ]:
#  Count Messages Per Day
day_count = {}
for msg in messages:
    date = msg["timestamp"].split(",")[0]
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1
print("DAILY MESSAGE COUNT")
print("=" * 50)
for date in day_count:
    print(
        date,
        ":",
        day_count[date],
        "messages"
    )
print("=" * 50)
print("Total Active Dates:", len(day_count))

DAILY MESSAGE COUNT
01/02/24 : 44 messages
02/02/24 : 47 messages
03/02/24 : 55 messages
04/02/24 : 53 messages
05/02/24 : 54 messages
06/02/24 : 49 messages
07/02/24 : 55 messages
08/02/24 : 53 messages
09/02/24 : 54 messages
10/02/24 : 46 messages
11/02/24 : 51 messages
12/02/24 : 48 messages
13/02/24 : 50 messages
14/02/24 : 52 messages
15/02/24 : 52 messages
16/02/24 : 54 messages
17/02/24 : 50 messages
18/02/24 : 52 messages
19/02/24 : 50 messages
20/02/24 : 57 messages
21/02/24 : 49 messages
22/02/24 : 52 messages
23/02/24 : 50 messages
24/02/24 : 52 messages
25/02/24 : 51 messages
26/02/24 : 50 messages
27/02/24 : 51 messages
28/02/24 : 51 messages
29/02/24 : 52 messages
01/03/24 : 52 messages
02/03/24 : 48 messages
03/03/24 : 51 messages
04/03/24 : 50 messages
05/03/24 : 52 messages
06/03/24 : 55 messages
07/03/24 : 52 messages
08/03/24 : 52 messages
09/03/24 : 46 messages
10/03/24 : 49 messages
11/03/24 : 53 messages
12/03/24 : 55 messages
13/03/24 : 51 messages
14/03/24 : 50 

In [ ]:
# Find Busiest Day
busiest_day = max(
    day_count,
    key=day_count.get
)
busiest_day_date = datetime.strptime(
    busiest_day,
    "%d/%m/%y"
)
print("BUSIEST DAY")
print("=" * 50)
print( "Date     :", busiest_day_date.strftime("%d %B %Y"))
print(  "Messages :",  day_count[busiest_day])
print("=" * 50)

BUSIEST DAY
Date     : 20 February 2024
Messages : 57


In [ ]:
# Count Messages Per Hour
hour_count = {}
# Initialize all 24 hours
for hour in range(24):
    hour_count[hour] = 0
# Count messages
for msg in messages:
    time = msg["timestamp"].split(", ")[1]
    hour = int(time.split(":")[0] )
    hour_count[hour] += 1
print("MESSAGES PER HOUR")
print("=" * 50)
for hour in range(24):
    next_hour = (hour + 1) % 24
    print( f"{hour:02d}:00 - {next_hour:02d}:00 :", hour_count[hour],"messages")
print("=" * 50)

MESSAGES PER HOUR
00:00 - 01:00 : 68 messages
01:00 - 02:00 : 87 messages
02:00 - 03:00 : 78 messages
03:00 - 04:00 : 100 messages
04:00 - 05:00 : 0 messages
05:00 - 06:00 : 0 messages
06:00 - 07:00 : 0 messages
07:00 - 08:00 : 0 messages
08:00 - 09:00 : 0 messages
09:00 - 10:00 : 0 messages
10:00 - 11:00 : 0 messages
11:00 - 12:00 : 427 messages
12:00 - 13:00 : 428 messages
13:00 - 14:00 : 115 messages
14:00 - 15:00 : 129 messages
15:00 - 16:00 : 118 messages
16:00 - 17:00 : 210 messages
17:00 - 18:00 : 178 messages
18:00 - 19:00 : 246 messages
19:00 - 20:00 : 154 messages
20:00 - 21:00 : 172 messages
21:00 - 22:00 : 210 messages
22:00 - 23:00 : 0 messages
23:00 - 00:00 : 347 messages


In [ ]:
# Find Busiest Hour
peak_hour = max(
    hour_count,
    key=hour_count.get
)
busiest_hour = peak_hour
next_hour = (
    peak_hour + 1
) % 24
print("BUSIEST HOUR")
print("=" * 50)
print("Time     :", f"{peak_hour:02d}:00 - {next_hour:02d}:00"
)
print( "Messages :",hour_count[peak_hour])
print("=" * 50)

BUSIEST HOUR
Time     : 12:00 - 13:00
Messages : 428


In [ ]:
# Feature 3 - Most Active Day and Hour Summary
print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)
print("Busiest Day  :", busiest_day_date.strftime("%d %B %Y"))
print("Day Messages :",day_count[busiest_day])
print()
print( "Busiest Hour :",  f"{peak_hour:02d}:00 - {next_hour:02d}:00")
print("Hour Messages:", hour_count[peak_hour])
print("=" * 60)

MOST ACTIVE DAY AND HOUR
Busiest Day  : 20 February 2024
Day Messages : 57

Busiest Hour : 12:00 - 13:00
Hour Messages: 428


Feature 4 – NumPy Activity Heatmap

In [ ]:
# Create Participant Index
sers = sorted(participants)
user_index = {}
for i, user in enumerate(sers):
    user_index[user] = i
print("HEATMAP PARTICIPANT INDEX")
print("=" * 60)
for user in sers:
    print( user,  ": Row",user_index[user] )
print("=" * 60)
print("Total Participants:", len(sers))

HEATMAP PARTICIPANT INDEX
Aman : Row 0
Anchal : Row 1
Karan : Row 2
Meera : Row 3
Naman : Row 4
Neha : Row 5
Priya : Row 6
Rahul : Row 7
Riya : Row 8
Vikas : Row 9
Total Participants: 10


In [ ]:
# Create 10 x 24 NumPy Activity Matrix
# Create 10 x 24 NumPy Activity Matrix
heatmap = np.zeros((len(participants), 24), dtype=int)
print("ACTIVITY MATRIX CREATED")
print("=" * 50)
print("Matrix Shape :", heatmap.shape)
print("Total Rows   :", heatmap.shape[0])
print("Total Columns:", heatmap.shape[1])
print("=" * 50)

ACTIVITY MATRIX CREATED
Matrix Shape : (10, 24)
Total Rows   : 10
Total Columns: 24


In [ ]:
# Fill NumPy Activity Matrix
for msg in messages:
    sender = msg["sender"]
    message_datetime = datetime.strptime(
        msg["timestamp"],"%d/%m/%y, %H:%M")
    hour = message_datetime.hour
    row_index = user_index[sender]
    heatmap[row_index][hour] += 1
print("Activity matrix filled successfully!")

Activity matrix filled successfully!


In [ ]:
# Display Numeric Activity Matrix

print("10 x 24 ACTIVITY MATRIX")
print("=" * 90)
for i in range(len(sers)):
    print(f"{sers[i]:<10}:", heatmap[i])
print("=" * 90)

10 x 24 ACTIVITY MATRIX
Aman      : [ 33  41  39  49   0   0   0   0   0   0   0   0   0   3   3   4  14   6
   0   0   0   0   0 165]
Anchal    : [ 0  0  0  0  0  0  0  0  0  0  0 80 76 11 15  8 14 12 24 17 20 29  0  0]
Karan     : [ 0  0  0  0  0  0  0  0  0  0  0  0  0 14 12 19 20 20 21 19 18 31  0  0]
Meera     : [ 0  0  0  0  0  0  0  0  0  0  0 33 36 23 27 21 26 28 48 31 32 39  0  0]
Naman     : [ 5 10  5 11  0  0  0  0  0  0  0 60 60 16 18 16 20 20 36 24 26 34  0 32]
Neha      : [ 0  0  0  0  0  0  0  0  0  0  0 34 29  7  9  5  6  8 18 11 16 15  0  0]
Priya     : [ 5  7  9 14  0  0  0  0  0  0  0 53 64 14 18 12 28 18 21 19 18 31  0 32]
Rahul     : [ 16  19  12  10   0   0   0   0   0   0   0 134 136  12  12  16  56  24
  60  20  32  12   0  73]
Riya      : [ 9 10 13 16  0  0  0  0  0  0  0 33 27 15 15 17 26 32 18 13 10 19  0 45]
Vikas     : [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 10  0  0  0  0  0  0]


In [ ]:
# Validate Heatmap Total
total_heatmap_messages = np.sum(heatmap)
print("HEATMAP VALIDATION")
print("=" * 50)
print("Messages in Heatmap :", total_heatmap_messages)
print("Parsed Messages     :",len(messages))
if total_heatmap_messages == len(messages):
    print( "Validation Status  : Successful")
else:
    print("Validation Status  : Failed")
print("=" * 50)

HEATMAP VALIDATION
Messages in Heatmap : 3067
Parsed Messages     : 3067
Validation Status  : Successful


In [ ]:
# Find Peak Hour Using NumPy
hour_totals = np.sum(
    heatmap,
    axis=0)
peak_hour = int(
    np.argmax(hour_totals))
next_hour = ( peak_hour + 1) % 24
print("NUMPY PEAK-HOUR ANALYSIS")
print("=" * 50)
print("Peak Hour :", f"{peak_hour:02d}:00 - {next_hour:02d}:00")
print( "Messages  :", hour_totals[peak_hour])
print("=" * 50)

NUMPY PEAK-HOUR ANALYSIS
Peak Hour : 12:00 - 13:00
Messages  : 428


In [ ]:
# Display Text-Based Activity Heatmap
# Display Text-Based Activity Heatmap
print("ACTIVITY HEATMAP")
print("=" * 90)
print("00 01 02 03 04 05 06 07 "
    "08 09 10 11 12 13 14 15 "
    "16 17 18 19 20 21 22 23")
for i in range(len(sers)):
    highest = np.max(
        heatmap[i])
    print(f"{sers[i]:<9}",end=" " )
    for hour in range(24):
        value = heatmap[i][hour]
        if highest == 0:
            symbol = "."
        else:
            activity_ratio = (value / highest)
            if activity_ratio <= 0.25:
                symbol = "."
            elif activity_ratio <= 0.50:
                symbol = "░"
            elif activity_ratio <= 0.75:
                symbol = "▒"
            else:
                symbol = "█"
        print( symbol,end="  ")
    print()
print("=" * 90)
print("Legend:")
print(". = 0% to 25% activity")
print("░ = 25% to 50% activity")
print("▒ = 50% to 75% activity")
print("█ = 75% to 100% activity")

ACTIVITY HEATMAP
00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman      .  .  .  ░  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  █  
Anchal    .  .  .  .  .  .  .  .  .  .  .  █  █  .  .  .  .  .  ░  .  .  ░  .  .  
Karan     .  .  .  .  .  .  .  .  .  .  .  .  .  ░  ░  ▒  ▒  ▒  ▒  ▒  ▒  █  .  .  
Meera     .  .  .  .  .  .  .  .  .  .  .  ▒  ▒  ░  ▒  ░  ▒  ▒  █  ▒  ▒  █  .  .  
Naman     .  .  .  .  .  .  .  .  .  .  .  █  █  ░  ░  ░  ░  ░  ▒  ░  ░  ▒  .  ▒  
Neha      .  .  .  .  .  .  .  .  .  .  .  █  █  .  ░  .  .  .  ▒  ░  ░  ░  .  .  
Priya     .  .  .  .  .  .  .  .  .  .  .  █  █  .  ░  .  ░  ░  ░  ░  ░  ░  .  ░  
Rahul     .  .  .  .  .  .  .  .  .  .  .  █  █  .  .  .  ░  .  ░  .  .  .  .  ▒  
Riya      .  .  ░  ░  .  .  .  .  .  .  .  ▒  ▒  ░  ░  ░  ▒  ▒  ░  ░  .  ░  .  █  
Vikas     .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  █  .  .  .  .  .  .  
Legend:
. = 0% to 25% activity
░ = 25% to 50% activity
▒ = 50% to 75% activity
█ 

Feature 5: Stop Words

In [ ]:
# Word Frequency Analysis

stop_words = [
    # Common English Stop Words
    "the", "is", "a", "an", "and", "or", "to",
    "of", "in", "on", "for", "at", "with",
    "from", "by", "as", "into", "about",
    "i", "you", "he", "she", "it", "we",
    "they", "this", "that", "these", "those",
    "are", "am", "was", "were", "be", "been",
    "have", "has", "had", "do", "does", "did",
    "will", "would", "can", "could", "should",
    "if", "but", "so", "because", "then",
    "also", "only", "very",

    # Common Hindi / Hinglish Stop Words
    "hai", "hain", "hu", "ho",
    "tha", "thi", "the",
    "ka", "ki", "ke", "ko",
    "me", "mein", "se", "pe", "par",
    "aur", "ya", "ye", "wo",
    "main", "hum", "tum", "aap",
    "mera", "meri", "mere",
    "tera", "teri", "tere",
    "apna", "apni", "apne",
    "sab", "koi", "kisi", "kuch",
    "kar", "karo", "karna", "karke",
    "raha", "rahi", "rahe",
    "gaya", "gayi", "gaye",
    "hoga", "hogi", "honge",
    "bhi", "hi", "ab", "abhi",
    "aaj", "kal", "phir",
    "ek", "time", "baje",
    "nahi", "nhi", "haan",
    "mat", "wala", "wali", "wale",
    "le", "lo", "do", "diya",
    "aa", "aana", "jana",
    "bol", "bata", "batao",
    "mujhe", "liye", "milega", "mil",
    "ne", "dekh"
]

print("Total Stop Words:",len(stop_words))

Total Stop Words: 137


In [ ]:
# Count Overall Word Frequency

word_count = {}

punctuation = (".,!?;:'\"()[]{}-_/@#$%^&*+=")
for msg in messages:
    text = msg["text"].lower()
    words = text.split()
    for word in words:
        # Remove punctuation
        word = word.strip( punctuation )
        # Skip empty words
        if word == "":
            continue
        # Skip single-character words
        if len(word) < 2:
            continue
        # Skip numeric values
        if word.isdigit():
            continue
        # Skip stop words
        if word in stop_words:
            continue
        # Count word frequency
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

print("WORD-FREQUENCY SUMMARY")
print("=" * 50)
print("Total Unique Words:", len(word_count))
print("=" * 50)

WORD-FREQUENCY SUMMARY
Total Unique Words: 323


In [ ]:
# Find Overall Top 10 Words

word_copy = word_count.copy()

top_words = []

for i in range(10):

    if len(word_copy) == 0:
        break

    top_word = max(
        word_copy,
        key=word_copy.get
    )

    top_word_count = word_copy[top_word]

    top_words.append(
        (top_word, top_word_count)
    )

    word_copy.pop(top_word)


print("=" * 60)
print("TOP 10 WORDS IN GROUP")
print("=" * 60)

for word, count in top_words:

    print(
        f"{word:<15} :",
        count
    )

print("=" * 60)

TOP 10 WORDS IN GROUP
please          : 509
bhej            : 329
kya             : 221
file            : 218
bhai            : 196
haha            : 161
attendance      : 159
output          : 158
error           : 150
yaar            : 145


In [ ]:
# Combined Word Frequency & Frequency Bars
word_count = {}
punctuation = ".,!?;:'\"()[]{}-_/@#$%^&*+="

for msg in messages:
    text = msg["text"].lower()
    words = text.split()
    for word in words:
        word = word.strip(punctuation)
        if word == "" or len(word) < 2 or word.isdigit() or word in stop_words:
            continue
        word_count[word] = word_count.get(word, 0) + 1
word_copy = word_count.copy()
top_words = []
for i in range(10):
    if len(word_copy) == 0:
        break
    top_word = max(word_copy, key=word_copy.get)
    top_words.append((top_word, word_copy[top_word]))
    word_copy.pop(top_word)
print("=" * 65)
print("TOP 10 WORDS WITH FREQUENCY BARS")
print("=" * 65)
if len(top_words) > 0:
    highest_count = top_words[0][1]
    for word, count in top_words:
        bar_length = int((count / highest_count) * 20)
        bar = "█" * bar_length
        print(f"{word:<15}", f"{bar:<20}", count)
else:
    print("No words found.")
print("=" * 65)

TOP 10 WORDS WITH FREQUENCY BARS
please          ████████████████████ 509
bhej            ████████████         329
kya             ████████             221
file            ████████             218
bhai            ███████              196
haha            ██████               161
attendance      ██████               159
output          ██████               158
error           █████                150
yaar            █████                145


In [ ]:
# Display Top 5 Words Per Person

print("=" * 60)
print("TOP 5 WORDS PER PERSON")
print("=" * 60)
for person in participants:
    person_word_count = {}
    for msg in messages:
        if msg["sender"] == person:
            text = msg["text"].lower()
            words = text.split()
            for word in words:
                word = word.strip(
                    punctuation
                )
                if word == "":
                    continue
                if len(word) < 2:
                    continue
                if word.isdigit():
                    continue
                if word in stop_words:
                    continue
                if word in person_word_count:
                    person_word_count[word] += 1
                else:
                    person_word_count[word] = 1
    word_copy = person_word_count.copy()
    print()
    print(person)
    print("-" * 30)
    for i in range(5):
        if len(word_copy) == 0:
            break
        top_word = max(
            word_copy,
            key=word_copy.get
        )
        print(
            f"{top_word:<15} :",
            word_copy[top_word]
        )
        word_copy.pop(top_word)
print("=" * 60)

TOP 5 WORDS PER PERSON

Anchal
------------------------------
bhejo           : 95
attendance      : 87
proxy           : 72
group           : 63
roll            : 60

Meera
------------------------------
please          : 214
bhej            : 213
photo           : 93
kya             : 65
yaar            : 65

Naman
------------------------------
kya             : 103
submission      : 61
compulsory      : 57
first           : 57
lecture         : 54

Neha
------------------------------
please          : 60
naam            : 36
miss            : 36
kernel          : 35
crash           : 35

Riya
------------------------------
haha            : 161
lol             : 122
coder           : 37
banne           : 37
side            : 37

Rahul
------------------------------
bhai            : 170
file            : 59
jaldi           : 57
bhej            : 57
proxy           : 51

Priya
------------------------------
please          : 211
pehle           : 59
lena            : 59
submit      

Feature 6 – Response Speed & Silent Streaks

In [ ]:
# Initialize Response-Time Data
response_data = {}
for person in participants:
    response_data[person] = []
print(
    "Response-time storage initialized "
    "for",
    len(response_data),
    "participants.")

Response-time storage initialized for 10 participants.


In [ ]:
# Calculate Response Times
for i in range(
    1,
    len(messages)
):
    current_sender = (
        messages[i]["sender"]
    )
    previous_sender = (
        messages[i - 1]["sender"]
    )
    # Response is counted only when sender changes
    if current_sender != previous_sender:
        current_time = datetime.strptime(
            messages[i]["timestamp"],
            "%d/%m/%y, %H:%M"
        )
        previous_time = datetime.strptime( messages[i - 1]["timestamp"], "%d/%m/%y, %H:%M"  )
        response_gap = ( current_time - previous_time )
        response_minutes = (response_gap.total_seconds() / 60)
        # Store only valid non-negative gaps
        if response_minutes >= 0:
            response_data[ current_sender].append( response_minutes )
print("Response times calculated successfully!")

Response times calculated successfully!


In [ ]:
# Calculate Average Response Time
average_response = {}
for person in participants:
    response_times = (
        response_data[person]
    )
    if len(response_times) > 0:
        average_response[person] = (sum(response_times) / len(response_times))
    else:
        average_response[person] = 0
print( "Average response times ""calculated successfully!")

Average response times calculated successfully!


In [ ]:
# Calculate and Display Average Response Times
average_response = {}
for person in participants:
    response_times = response_data[person]
    if len(response_times) > 0:
        average_response[person] = sum(response_times) / len(response_times)
    else:
        average_response[person] = 0

print("=" * 60)
print("AVERAGE RESPONSE TIME")
print("=" * 60)
for person in participants:
    print(f"{person:<12} :",
        f"{average_response[person]:.2f}",
        "minutes" )
print("=" * 60)

AVERAGE RESPONSE TIME
Anchal       : 38.39 minutes
Meera        : 21.99 minutes
Naman        : 68.93 minutes
Neha         : 2.16 minutes
Riya         : 1.97 minutes
Rahul        : 33.80 minutes
Priya        : 26.59 minutes
Aman         : 41.05 minutes
Karan        : 1.86 minutes
Vikas        : 68.80 minutes


In [ ]:
# Find Fastest and Slowest Responder
valid_average_response = {}
for person in participants:
    if len(response_data[person]) > 0:

        valid_average_response[person] = (average_response[person])
if len(valid_average_response) > 0:
    fastest_responder = min(valid_average_response,
        key=valid_average_response.get
    )
    slowest_responder = max(
        valid_average_response,
        key=valid_average_response.get
    )
    print("FASTEST AND SLOWEST RESPONDER")
    print("=" * 60)
    print(
        "Fastest Responder :",
        fastest_responder,
        "-",
        f"{average_response[fastest_responder]:.2f}",
        "minutes"
    )
    print("Slowest Responder :",slowest_responder,
        "-",f"{average_response[slowest_responder]:.2f}",
        "minutes"
    )
    print("=" * 60)
else:
    fastest_responder = "Not Found"
    slowest_responder = "Not Found"
    print("Response data not available.")

FASTEST AND SLOWEST RESPONDER
Fastest Responder : Karan - 1.86 minutes
Slowest Responder : Naman - 68.93 minutes


In [ ]:
# Collect Active Dates Per Person
active_dates = {}
for person in participants:
    active_dates[person] = []
for msg in messages:
    sender = msg["sender"]
    message_datetime = datetime.strptime(
        msg["timestamp"],
        "%d/%m/%y, %H:%M"
    )
    message_date = (
        message_datetime.date()
    )
    if message_date not in active_dates[sender]:
        active_dates[sender].append(
            message_date
        )
print("ACTIVE DAYS PER PERSON")
print("=" * 50)
for person in participants:
    print(f"{person:<12} :", len(active_dates[person]), "active days")
print("=" * 50)

ACTIVE DAYS PER PERSON
Anchal       : 60 active days
Meera        : 60 active days
Naman        : 60 active days
Neha         : 60 active days
Riya         : 60 active days
Rahul        : 60 active days
Priya        : 60 active days
Aman         : 60 active days
Karan        : 60 active days
Vikas        : 10 active days


In [ ]:
# Calculate Longest Silent Streak
silent_streaks = {}
for person in participants:
    dates = active_dates[person]
    # Sort active dates
    dates.sort()
    # Person never sent any message
    if len(dates) == 0:
        silent_streaks[person] = (total_days)
        continue
    # Silence before first active date
    beginning_silence = (
        dates[0] - chat_start_date
    ).days
    longest_streak = (beginning_silence)
    # Silence between active dates
    for i in range(
        1,
        len(dates)
    ):
        gap = (
            dates[i] -
            dates[i - 1]
        ).days - 1
        if gap > longest_streak:
            longest_streak = gap
    # Silence after last active date
    ending_silence = (
        chat_end_date - dates[-1] ).days
    if ending_silence > longest_streak:
        longest_streak = (ending_silence)
    silent_streaks[person] = ( longest_streak )
print("Silent streaks calculated successfully!")

Silent streaks calculated successfully!


In [ ]:
# Display Longest Silent Streaks

print("=" * 60)
print("LONGEST SILENT STREAKS")
print("=" * 60)

for person in participants:

    print(
        f"{person:<12} :",
        silent_streaks[person],
        "days"
    )

print("=" * 60)

LONGEST SILENT STREAKS
Anchal       : 0 days
Meera        : 0 days
Naman        : 0 days
Neha         : 0 days
Riya         : 0 days
Rahul        : 0 days
Priya        : 0 days
Aman         : 0 days
Karan        : 0 days
Vikas        : 6 days


In [ ]:
# Feature 6 - Final Summary
print("=" * 65)
print("RESPONSE SPEED AND SILENT-STREAK SUMMARY")
print("=" * 65)
print(
    "Fastest Responder :",
    fastest_responder,
    "-",
    f"{average_response[fastest_responder]:.2f}",
    "minutes"
)
print(
    "Slowest Responder :",
    slowest_responder,
    "-",
    f"{average_response[slowest_responder]:.2f}",
    "minutes"
)
print()
print("LONGEST SILENT STREAK PER PERSON")
print("-" * 65)
for person in participants:
    print( f"{person:<12} : "
        f"{silent_streaks[person]} days"  )
print("=" * 65)

RESPONSE SPEED AND SILENT-STREAK SUMMARY
Fastest Responder : Karan - 1.86 minutes
Slowest Responder : Naman - 68.93 minutes

LONGEST SILENT STREAK PER PERSON
-----------------------------------------------------------------
Anchal       : 0 days
Meera        : 0 days
Naman        : 0 days
Neha         : 0 days
Riya         : 0 days
Rahul        : 0 days
Priya        : 0 days
Aman         : 0 days
Karan        : 0 days
Vikas        : 6 days


Feature 7 – Personality Archetype Detection

In [ ]:
# THE GROUP MOM

group_mom_words = [
    "okay",
    "safe",
    "eat",
    "sleep",
    "take care",
    "are you",
    "please",
    "reminder",
    "drink water",
    "don't forget",
    "rest"
]
mom_count = {}
for person in participants:
    mom_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()
    is_caring_message = False
    for phrase in group_mom_words:
        if phrase in text:
            is_caring_message = True
    # Count each message only once
    if is_caring_message:
        mom_count[sender] += 1
group_mom = max(
    mom_count,
    key=mom_count.get
)
print("Group Mom :",group_mom)

Group Mom : Priya


In [ ]:
# THE STORYTELLER
storyteller = max(
    story_average,
    key=story_average.get
)
print("Storyteller :",storyteller)
print("Average Words Per Message :", round(story_average[storyteller], 2
    )
)

Storyteller : Karan
Average Words Per Message : 32.6


In [ ]:
# THE DRAMA QUEEN
drama_count = {}
for person in participants:
    drama_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].strip()
    is_dramatic_message = False
    # Check ALL-CAPS messages
    if (text.isupper()and len(text) >= 3):
        is_dramatic_message = True
    # Check multiple exclamation marks
    if text.count("!") >= 2:
        is_dramatic_message = True
    # Count each message only once
    if is_dramatic_message:
        drama_count[sender] += 1
drama_queen = max(drama_count,
    key=drama_count.get
)
print("Drama Queen :",drama_queen)

Drama Queen : Neha


In [ ]:
# THE SPAMMER
burst_total = {}
burst_count = {}
for person in participants:
    burst_total[person] = 0
    burst_count[person] = 0
current_sender = ""
current_burst = 0
for msg in messages:
    sender = msg["sender"]
    if sender == current_sender:
        current_burst += 1
    else:
        if current_sender != "":
            burst_total[ current_sender
            ] += current_burst
            burst_count[
                current_sender
            ] += 1
        current_sender = sender
        current_burst = 1
# Store final message burst
if current_sender != "":
    burst_total[
        current_sender
    ] += current_burst
    burst_count[
        current_sender
    ] += 1
# Calculate average burst
average_burst = {}
for person in participants:
    if burst_count[person] > 0:
        average_burst[person] = (
            burst_total[person]
            / burst_count[person] )
    else:
        average_burst[person] = 0
spammer = max(
    average_burst,
    key=average_burst.get
)
print( "Spammer :",spammer)
print("Average Burst :", round( average_burst[spammer], 2
    )
)

Spammer : Rahul
Average Burst : 3.3


In [ ]:
# THE GHOST
silent_day_percentage = {}
for person in participants:
    active_day_count = len(
        active_dates[person]
    )
    silent_days = ( total_days - active_day_count)
    percentage = (silent_days / total_days) * 100
    silent_day_percentage[person] = (percentage)
ghost = max(silent_day_percentage,key=silent_day_percentage.get)
print("Ghost :", ghost
)
print( "Silent Days Percentage :",
    round(silent_day_percentage[ghost], 2 ),
    "%"
)

Ghost : Vikas
Silent Days Percentage : 83.33 %


In [ ]:
# THE NIGHT OWL
night_count = {}
for person in participants:
    night_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    message_datetime = datetime.strptime(
        msg["timestamp"],
        "%d/%m/%y, %H:%M"
    )
    hour = message_datetime.hour
    # Official night range: 23:00 to 04:59
    if hour >= 23 or hour <= 4:
        night_count[sender] += 1
night_message_percentage = {}
for person in participants:
    if message_count[person] > 0:
        night_message_percentage[person] = (night_count[person] / message_count[person] ) * 100
    else:
        night_message_percentage[person] = 0
night_owl = max(night_message_percentage, key=night_message_percentage.get
)
print("Night Owl :",night_owl)
print("Late-Night Messages :",round(night_message_percentage[night_owl], 2),"%"
)

Night Owl : Aman
Late-Night Messages : 91.6 %


In [ ]:
# THE COMEDIAN
comedy_words = [
    "haha",
    "lol",
    "lmao",
    "rofl",
    "lmfao"
]
comedy_count = {}
for person in participants:
    comedy_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()
    is_comedy_message = False
    for word in comedy_words:
        if word in text:
            is_comedy_message = True
    # Count each message only once
    if is_comedy_message:comedy_count[sender] += 1
comedy_message_percentage = {}
for person in participants:
    if message_count[person] > 0:
        comedy_message_percentage[person] = (comedy_count[person]/ message_count[person]) * 100
    else:
        comedy_message_percentage[person] = 0
comedian = max(
    comedy_message_percentage,
    key=comedy_message_percentage.get
)
print("Comedian :",comedian
)
print("Comedy Messages :",
    round(    comedy_message_percentage[comedian],2),
    "%"
)

Comedian : Riya
Comedy Messages : 100.0 %


In [ ]:
# THE QUESTION MASTER
question_count = {}
for person in participants:
    question_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].strip()
    if text.endswith("?"):
        question_count[sender] += 1
question_message_percentage = {}
for person in participants:
    if message_count[person] > 0:
        question_message_percentage[person] = (
            question_count[person]
            / message_count[person]
        ) * 100
    else:
        question_message_percentage[person] = 0
question_master = max(
    question_message_percentage,
    key=question_message_percentage.get
)
print(
    "Question Master :",
    question_master
)
print(
    "Question Messages :",
    round(
        question_message_percentage[
            question_master
        ],
        2
    ),
    "%"
)

Question Master : Naman
Question Messages : 100.0 %


In [ ]:
# THE PROXY COORDINATOR

proxy_words = [
    "proxy",
    "attendance",
    "roll number",
    "present",
    "attendance short"
]
proxy_count = {}
for person in participants:
    proxy_count[person] = 0
for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()
    is_proxy_message = False
    for phrase in proxy_words:
        if phrase in text:
            is_proxy_message = True
    # Count each message only once
    if is_proxy_message:
        proxy_count[sender] += 1
proxy_message_percentage = {}
for person in participants:
    if message_count[person] > 0:
        proxy_message_percentage[person] = (
            proxy_count[person]
            / message_count[person]
        ) * 100
    else:
        proxy_message_percentage[person] = 0
proxy_coordinator = max(
    proxy_message_percentage,
    key=proxy_message_percentage.get
)
print(
    "Proxy Coordinator :",
    proxy_coordinator
)

print(
    "Proxy-Related Messages :",
    round(
        proxy_message_percentage[
            proxy_coordinator
        ],
        2
    ),
    "%"
)

Proxy Coordinator : Anchal
Proxy-Related Messages : 40.2 %


In [ ]:
# THE ASSIGNMENT BEGGAR
assignment_words = [
    "assignment",
    "notes",
    "bhej do",
    "send karo",
    "photo bhej",
    "pdf forward",
    "notebook bhej",
    "format bhej",
    "viva questions",
    "practical file",
    "index",
    "certificate page",
    "resume format",
    "dashboard file"
]

assignment_count = {}
for person in participants:
    assignment_count[person] = 0

for msg in messages:
    sender = msg["sender"]
    text = msg["text"].lower()

    # Initialize flag for each message
    is_assignment_message = False

    for phrase in assignment_words:
        if phrase in text:
            is_assignment_message = True

    # Count each message only once if flagged
    if is_assignment_message:
        assignment_count[sender] += 1

assignment_message_percentage = {}
for person in participants:
    if message_count[person] > 0:
        assignment_message_percentage[person] = (
            assignment_count[person]
            / message_count[person]
        ) * 100
    else:
        assignment_message_percentage[person] = 0

assignment_beggar = max(
    assignment_message_percentage,
    key=assignment_message_percentage.get
)

print(
    "Assignment Beggar :",
    assignment_beggar
)

print(
    "Assignment-Request Messages :",
    round(
        assignment_message_percentage[
            assignment_beggar
        ],
        2
    ),
    "%"
)

Assignment Beggar : Meera
Assignment-Request Messages : 89.53 %


In [ ]:
# Feature 7 - Final Personality Archetypes

night_percentage = (
    night_message_percentage[
        night_owl
    ]
)

mom_percentage = ( mom_count[group_mom] / message_count[group_mom]) * 100
drama_percentage = (drama_count[drama_queen]  / message_count[drama_queen]) * 100
comedy_percentage = ( comedy_message_percentage[ comedian])
question_percentage = ( question_message_percentage[question_master])
ghost_silent_percentage = ( silent_day_percentage[ ghost])
proxy_percentage = ( proxy_message_percentage[ proxy_coordinator ])
assignment_percentage = ( assignment_message_percentage[ assignment_beggar])

print("=" * 75)
print("PERSONALITY ARCHETYPES")
print("=" * 75)
print( f"{spammer:<12} → THE SPAMMER "
    f"(avg {average_burst[spammer]:.2f} messages in a row)")

print(f"{group_mom:<12} → THE GROUP MOM "
    f"({mom_percentage:.2f}% caring messages)")

print(f"{night_owl:<12} → THE NIGHT OWL "
    f"({night_percentage:.2f}% late-night messages)")

print(f"{storyteller:<12} → THE STORYTELLER "
    f"(avg {story_average[storyteller]:.2f} words per message)")

print(f"{drama_queen:<12} → THE DRAMA QUEEN "
    f"({drama_percentage:.2f}% dramatic messages)")

print(f"{ghost:<12} → THE GHOST "
    f"(silent {ghost_silent_percentage:.2f}% of days)")
print(f"{comedian:<12} → THE COMEDIAN "
    f"({comedy_percentage:.2f}% comedy messages)")

print(f"{question_master:<12} → THE QUESTION MASTER "
    f"({question_percentage:.2f}% question messages)")

print(f"{proxy_coordinator:<12} → THE PROXY COORDINATOR "
    f"({proxy_percentage:.2f}% proxy-related messages)")

print(f"{assignment_beggar:<12} → THE ASSIGNMENT BEGGAR "
    f"({assignment_percentage:.2f}% assignment-request messages)")

print("=" * 75)

PERSONALITY ARCHETYPES
Rahul        → THE SPAMMER (avg 3.30 messages in a row)
Priya        → THE GROUP MOM (100.00% caring messages)
Aman         → THE NIGHT OWL (91.60% late-night messages)
Karan        → THE STORYTELLER (avg 32.60 words per message)
Neha         → THE DRAMA QUEEN (100.00% dramatic messages)
Vikas        → THE GHOST (silent 83.33% of days)
Riya         → THE COMEDIAN (100.00% comedy messages)
Naman        → THE QUESTION MASTER (100.00% question messages)
Anchal       → THE PROXY COORDINATOR (40.20% proxy-related messages)
Meera        → THE ASSIGNMENT BEGGAR (89.53% assignment-request messages)


Feature 8 – Final Report

In [ ]:
#  Final GroupDNA Report

report_next_hour = (
    peak_hour + 1
) % 24


print("=" * 75)
print(" GROUPDNA FINAL REPORT")
print("CSE 2026 - CODE, CANTEEN & CHAOS")
print("=" * 75)


# Chat Summary

print("\nCHAT SUMMARY")
print("-" * 75)

print("Dataset Period     :",
    first_date.strftime("%d %B %Y"),
    "to",
    last_date.strftime("%d %B %Y")
)

print("Total Days         :", total_days)
print("Total Raw Lines    :", len(lines))
print("Real Messages      :", len(messages))
print("System Messages    :", system_messages)
print("Media Omitted      :", media_omitted)
print("Deleted Messages   :", deleted_messages)
print("Continuation Lines :", continuation_lines)
print("Total Members      :", len(participants))


# Activity Summary

print("\nACTIVITY SUMMARY")
print("-" * 75)

print("Most Active Person  :",
    most_active,
    "-",
    message_count[most_active],
    "messages")

print( "Least Active Person :",
    least_active,
    "-",
    message_count[least_active],
    "messages")

print( "Busiest Day         :",
    busiest_day_date.strftime("%d %B %Y"),
    "-",
    day_count[busiest_day],
    "messages")

print("Busiest Hour        :",
    f"{peak_hour:02d}:00 - "
    f"{report_next_hour:02d}:00",
    "-",
    hour_totals[peak_hour],
    "messages")

# Group Participation
print("\nGROUP PARTICIPATION")
print("-" * 75)

for person, count in sorted_message_count:

    participation_percentage = (
        count / len(messages)
    ) * 100

    print(f"{person:<12} : "
        f"{count:<4} messages "
        f"({participation_percentage:.2f}%)")


# Top Words

print("\nTOP 10 WORDS")
print("-" * 75)

if len(top_words) > 0:
    highest_word_count = (
        top_words[0][1]
    )
    for word, count in top_words:
        bar_length = int(
            (
                count /
                highest_word_count
            ) * 20
        )
        bar = "█" * bar_length
        print( f"{word:<15}",
            f"{bar:<20}",
            count
        )
else:
    print("No words found.")
# Response Analysis
print("\nRESPONSE ANALYSIS")
print("-" * 75)
print(
    "Fastest Replier :",
    fastest_responder,
    "-",
    f"{average_response[fastest_responder]:.2f}",
    "minutes"
)
print(
    "Slowest Replier :",
    slowest_responder,
    "-",
    f"{average_response[slowest_responder]:.2f}",
    "minutes"
)
# Personality Archetypes
print("\nPERSONALITY ARCHETYPES")
print("-" * 75)
print( f"{spammer:<12} → THE SPAMMER "
    f"(avg {average_burst[spammer]:.2f} messages in a row)"
)

print( f"{group_mom:<12} → THE GROUP MOM "
    f"({mom_percentage:.2f}% caring messages)")

print(f"{night_owl:<12} → THE NIGHT OWL "
    f"({night_percentage:.2f}% late-night messages)")

print( f"{storyteller:<12} → THE STORYTELLER "
    f"(avg {story_average[storyteller]:.2f} words per message)")

print(f"{drama_queen:<12} → THE DRAMA QUEEN "
    f"({drama_percentage:.2f}% dramatic messages)")

print(f"{ghost:<12} → THE GHOST "
    f"(silent {ghost_silent_percentage:.2f}% of days)")

print(f"{comedian:<12} → THE COMEDIAN "
    f"({comedy_percentage:.2f}% comedy messages)")

print(f"{question_master:<12} → THE QUESTION MASTER "
    f"({question_percentage:.2f}% question messages)")

print(f"{proxy_coordinator:<12} → THE PROXY COORDINATOR "
    f"({proxy_percentage:.2f}% proxy-related messages)")

print( f"{assignment_beggar:<12} → THE ASSIGNMENT BEGGAR "
    f"({assignment_percentage:.2f}% assignment-request messages)")


# Silent Streaks
print("\nLONGEST SILENT STREAKS")
print("-" * 75)
for person in participants:
    print(f"{person:<12} : "
        f"{silent_streaks[person]} days")
# Heatmap Validation
print("\nNUMPY HEATMAP VALIDATION")
print("-" * 75)
print("Heatmap Shape    :",
    heatmap.shape
)
print("Heatmap Messages :", total_heatmap_messages)
print("Parsed Messages  :",len(messages))

if total_heatmap_messages == len(messages):
    print("Validation Status: Successful")
else:
    print("Validation Status: Failed" )


# End of Report

print("\n" + "=" * 75)
print("                     GENERATED BY GROUPDNA")
print("                  BUILT WITH PYTHON + NUMPY")
print("=" * 75)

## Conclusion

The **GroupDNA – CSE 2026: Code, Canteen & Chaos** project analyzes a synthetic Indian college WhatsApp group chat using Python fundamentals, NumPy, File I/O and Datetime.

The dataset covers **60 days**, from **01 February 2024 to 31 March 2024**, and includes:

* 3,125 raw lines
* 3,067 text messages
* 10 participants
* 11 system messages
* 30 media entries
* 15 deleted messages
* 2 multi-line continuation entries

The analysis showed that **Rahul** was the most active participant, while **Vikas** was the least active. The busiest day was **20 February 2024**, with 57 messages, and the busiest hour was **12:00–13:00**, with 428 messages.

Each participant was also assigned a personality archetype:

* Rahul — The Spammer
* Priya — The Group Mom
* Aman — The Night Owl
* Karan — The Storyteller
* Neha — The Drama Queen
* Vikas — The Ghost
* Riya — The Comedian
* Naman — The Question Master
* Anchal — The Proxy Coordinator
* Meera — The Assignment Beggar

This project helped me understand how raw chat data can be cleaned, analyzed and converted into meaningful insights using basic Python and NumPy.

---

## Reflection

### Challenges

The main challenge was parsing different types of chat entries, including system messages, deleted messages, media entries and multi-line messages.

Calculating response times, silent streaks and personality percentages correctly was also challenging.

### What I Learned

Through this project, I learned:

* File handling in Python
* String parsing and text cleaning
* Working with lists and dictionaries
* Using loops and conditional statements
* Datetime calculations
* Word-frequency analysis
* NumPy matrix operations
* Creating a text-based activity heatmap
* Finding response patterns and silent streaks
* Detecting behavioural patterns using rules
* Validating analytical results

### Future Improvements

In the future, I would like to:

* Use Matplotlib and Seaborn to create graphical charts and heatmaps
* Build an interactive Power BI dashboard
* Add emoji-frequency analysis
* Add sentiment analysis

---


